In [1]:
# CELL 1 — Install dependencies

!pip install -q -U transformers peft accelerate bitsandbytes datasets huggingface_hub trl
!pip install -q -U sentencepiece safetensors protobuf rdkit tqdm

import os
import gc
import json
import random
import getpass
from pathlib import Path
from typing import Any, Dict, Optional

import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
else:
    raise RuntimeError("CUDA is False. Enable GPU runtime before training.")

Torch: 2.8.0+cu128
CUDA: True
GPU: NVIDIA L40S
CUDA capability: (8, 9)


In [2]:
# CELL 2 — Login and config

from huggingface_hub import login, whoami, HfApi, snapshot_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste Hugging Face token with write access: ")

login(token=HF_TOKEN)
HF_USERNAME = whoami(token=HF_TOKEN)["name"]

print("Logged in as:", HF_USERNAME)


# Model / project

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
HF_MODEL_REPO_NAME = "ord-retro-qwen25-15b"

LORA_REPO_ID = f"{HF_USERNAME}/{HF_MODEL_REPO_NAME}-lora"
MERGED_REPO_ID = f"{HF_USERNAME}/{HF_MODEL_REPO_NAME}-merged-fp16"


# Paths

WORK_DIR = Path("/content/ord_retro_training")
DATA_DIR = WORK_DIR / "prepared_data"
OUTPUT_DIR = WORK_DIR / "trainer_output"
ADAPTER_DIR = WORK_DIR / "lora_adapter"
MERGED_DIR = WORK_DIR / "merged_fp16"

TRAIN_JSONL = DATA_DIR / "train_sft.jsonl"
EVAL_JSONL = DATA_DIR / "eval_sft.jsonl"

for p in [WORK_DIR, DATA_DIR, OUTPUT_DIR, ADAPTER_DIR, MERGED_DIR]:
    p.mkdir(parents=True, exist_ok=True)


# Training config

MAX_SEQ_LENGTH = 2048

MAX_STEPS = 2500

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRAD_ACCUM = 8

LEARNING_RATE = 2e-4
SEED = 3407

USE_BF16 = True
USE_FP16 = False

random.seed(SEED)

print("LoRA repo:", LORA_REPO_ID)
print("Merged repo:", MERGED_REPO_ID)
print("Work dir:", WORK_DIR)

Paste Hugging Face token with write access:  ········


Logged in as: oleh13
LoRA repo: oleh13/ord-retro-qwen25-15b-lora
Merged repo: oleh13/ord-retro-qwen25-15b-merged-fp16
Work dir: /content/ord_retro_training


In [5]:
# CELL 3 — Download ORD from Hugging Face and build SFT JSONL
!python -m pip install -q -U ord-schema protobuf

import os
import gc
import json
import gzip
import random
import shutil
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

from tqdm.auto import tqdm

from huggingface_hub import HfApi, hf_hub_download

from rdkit import Chem
from rdkit import RDLogger, rdBase

from google.protobuf.json_format import MessageToDict
from ord_schema.proto import dataset_pb2


# RDKit logging
RDLogger.DisableLog("rdApp.warning")
RDLogger.DisableLog("rdApp.error")
rdBase.DisableLog("rdApp.warning")
rdBase.DisableLog("rdApp.error")


# Dataset config

HF_DATASET_ID = "open-reaction-database/ord-data"

MAX_PB_FILES = 120

MAX_TOTAL_EXAMPLES = 30000
MAX_EVAL_EXAMPLES = 800

DEDUP_BY_PRODUCT = True

USE_REACTANTS_IN_PROMPT = False

FORCE_REBUILD_DATASET = True

SEED = 3407
random.seed(SEED)


# Paths

WORK_DIR = Path("/content/ord_retro_training")
ORD_DIR = WORK_DIR / "ord_data"
DATA_DIR = WORK_DIR / "prepared_data"

TRAIN_JSONL = DATA_DIR / "train_sft.jsonl"
EVAL_JSONL = DATA_DIR / "eval_sft.jsonl"
ROWS_JSONL = DATA_DIR / "extracted_rows.jsonl"

ORD_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)


# Prompt

SYSTEM_PROMPT_TRAIN = """
You are an expert Organic Chemist specializing in Retrosynthesis.
Your task is to propose practical single-step forward synthesis routes that make the given target molecule.

Rules:
1. Output valid JSON only.
2. Generate exactly 1 route.
3. Each route must contain exactly 1 reaction step.
4. The step product_smiles must equal the target molecule SMILES.
5. Use standard SMILES strings.
6. Use realistic organic chemistry conditions.
7. Do not output Markdown.
8. If evidence reaction IDs are known, include them. Do not invent unsupported evidence.
""".strip()


# Basic chemistry helpers

def canonical_smiles(smiles: Optional[str]) -> Optional[str]:
    if not smiles:
        return None

    smiles = str(smiles).strip()

    if not smiles:
        return None

    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        return Chem.MolToSmiles(mol, canonical=True)

    except Exception:
        return None


def unique_keep_order(values: Iterable[str]) -> List[str]:
    seen = set()
    result = []

    for value in values:
        if value is None:
            continue

        value = str(value).strip()

        if not value or value.lower() in {"none", "null", "nan", "not specified"}:
            continue

        if value not in seen:
            seen.add(value)
            result.append(value)

    return result


def first_identifier_value(obj: Dict[str, Any], identifier_type: str) -> Optional[str]:
    target = identifier_type.upper()

    for ident in obj.get("identifiers", []) or []:
        if str(ident.get("type", "")).upper() == target:
            value = ident.get("value")

            if value:
                return str(value).strip()

    return None


def first_smiles(obj: Dict[str, Any]) -> Optional[str]:
    return canonical_smiles(first_identifier_value(obj, "SMILES"))


def first_name(obj: Dict[str, Any]) -> Optional[str]:
    return first_identifier_value(obj, "NAME")


def format_quantity(q: Optional[Dict[str, Any]]) -> str:
    if not isinstance(q, dict):
        return "not specified"

    value = q.get("value")
    units = q.get("units")

    if value is None:
        return "not specified"

    if units:
        return f"{value} {str(units).lower()}"

    return str(value)


# ORD extraction helpers

def extract_temperature(reaction: Dict[str, Any]) -> str:
    try:
        return format_quantity(reaction["conditions"]["temperature"]["setpoint"])
    except Exception:
        return "not specified"


def extract_pressure(reaction: Dict[str, Any]) -> str:
    try:
        return format_quantity(reaction["conditions"]["pressure"]["setpoint"])
    except Exception:
        return "not specified"


def extract_atmosphere(reaction: Dict[str, Any]) -> str:
    environment = reaction.get("setup", {}).get("environment", {})

    details = environment.get("details")
    env_type = environment.get("type")

    if details:
        return str(details)

    if env_type:
        return str(env_type).lower()

    return "not specified"


def extract_workup(reaction: Dict[str, Any]) -> str:
    parts = []

    for workup in reaction.get("workups", []) or []:
        workup_type = workup.get("type")
        details = workup.get("details")

        if workup_type and details:
            parts.append(f"{workup_type}: {details}")
        elif details:
            parts.append(str(details))
        elif workup_type:
            parts.append(str(workup_type))

    return "; ".join(parts) if parts else "not specified"


def extract_reaction_time(outcome: Dict[str, Any]) -> str:
    return format_quantity(outcome.get("reaction_time"))


def extract_yield_percent(product: Dict[str, Any]) -> Optional[float]:
    yields = []

    for measurement in product.get("measurements", []) or []:
        if str(measurement.get("type", "")).upper() != "YIELD":
            continue

        percentage = measurement.get("percentage") or {}
        value = percentage.get("value")

        if value is None:
            continue

        try:
            yields.append(float(value))
        except Exception:
            pass

    if not yields:
        return None

    return max(yields)


def extract_inputs(reaction: Dict[str, Any]) -> Dict[str, List[str]]:
    reactants = []
    solvents = []
    reagents = []

    inputs = reaction.get("inputs") or {}

    for _, reaction_input in inputs.items():
        for component in reaction_input.get("components", []) or []:
            role = str(component.get("reaction_role", "")).upper().strip()

            smiles = first_smiles(component)
            name = first_name(component)
            label = name or smiles

            if role in {"REACTANT", "STARTING_MATERIAL", "SUBSTRATE"}:
                if smiles:
                    reactants.append(smiles)

            elif role == "SOLVENT":
                if label:
                    solvents.append(label)

            elif role in {"INTERNAL_STANDARD", "PRODUCT"}:
                continue

            else:
                # Reaction role metadata.
                if label:
                    reagents.append(label)

    return {
        "reactants": unique_keep_order(reactants),
        "solvents": unique_keep_order(solvents),
        "reagents": unique_keep_order(reagents),
    }


def extract_best_product(reaction: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    candidates = []

    for outcome in reaction.get("outcomes", []) or []:
        reaction_time = extract_reaction_time(outcome)

        for product in outcome.get("products", []) or []:
            role = str(product.get("reaction_role", "")).upper().strip()

            if role and role != "PRODUCT":
                continue

            product_smiles = first_smiles(product)

            if not product_smiles:
                continue

            y = extract_yield_percent(product)
            desired = bool(product.get("is_desired_product", False))

            score = 0.0

            if desired:
                score += 10000.0

            if y is not None:
                score += 1000.0 + y

            candidates.append(
                {
                    "product_smiles": product_smiles,
                    "yield_percent": y,
                    "reaction_time": reaction_time,
                    "is_desired_product": desired,
                    "score": score,
                }
            )

    if not candidates:
        return None

    candidates.sort(key=lambda x: x["score"], reverse=True)

    return candidates[0]


def extract_reaction_row(reaction: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    reaction_id = str(reaction.get("reaction_id") or reaction.get("id") or "")

    inputs = extract_inputs(reaction)
    product = extract_best_product(reaction)

    if product is None:
        return None

    reactants = inputs["reactants"]
    product_smiles = product["product_smiles"]

    if not reactants or not product_smiles:
        return None

    temperature = extract_temperature(reaction)
    pressure = extract_pressure(reaction)
    atmosphere = extract_atmosphere(reaction)
    workup = extract_workup(reaction)

    condition_details = []

    conditions = reaction.get("conditions") or {}

    if conditions.get("details"):
        condition_details.append(str(conditions["details"]))

    if pressure != "not specified":
        condition_details.append(f"pressure: {pressure}")

    important_conditions = "; ".join(condition_details) if condition_details else "not specified"

    row = {
        "reaction_id": reaction_id,
        "reactants": reactants,
        "product_smiles": product_smiles,
        "reagents": inputs["reagents"],
        "solvents": inputs["solvents"],
        "temperature_celsius": temperature,
        "reaction_time": product["reaction_time"],
        "yield_percent": product["yield_percent"],
        "atmosphere": atmosphere,
        "workup_purification": workup,
        "important_conditions": important_conditions,
    }

    has_useful_label = (
        bool(row["reagents"])
        or bool(row["solvents"])
        or row["temperature_celsius"] != "not specified"
        or row["reaction_time"] != "not specified"
        or row["yield_percent"] is not None
    )

    if not has_useful_label:
        return None

    return row


def iter_ord_reactions_from_pb_gz(pb_gz_path: Path) -> Iterable[Dict[str, Any]]:
    dataset = dataset_pb2.Dataset()

    with gzip.open(pb_gz_path, "rb") as f:
        dataset.ParseFromString(f.read())

    for reaction_proto in dataset.reactions:
        reaction_dict = MessageToDict(
            reaction_proto,
            preserving_proto_field_name=True,
        )

        yield reaction_dict


def quality_score(row: Dict[str, Any]) -> float:
    score = 0.0

    score += 10.0 * len(row["reactants"])
    score += 5.0 * len(row["reagents"])
    score += 5.0 * len(row["solvents"])

    if row["temperature_celsius"] != "not specified":
        score += 5.0

    if row["reaction_time"] != "not specified":
        score += 5.0

    if row["yield_percent"] is not None:
        score += 20.0 + float(row["yield_percent"])

    if row["reaction_id"]:
        score += 2.0

    return score


def deduplicate_rows_by_product(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    best_by_product = {}

    for row in rows:
        product = row["product_smiles"]

        if product not in best_by_product:
            best_by_product[product] = row

        elif quality_score(row) > quality_score(best_by_product[product]):
            best_by_product[product] = row

    return list(best_by_product.values())


# SFT formatting

def build_stoichiometry(row: Dict[str, Any]) -> str:
    if not row["reactants"]:
        return "not specified"

    parts = [
        f"Reactant {i + 1}: 1.0 equiv"
        for i, _ in enumerate(row["reactants"])
    ]

    if row["reagents"]:
        parts.append("reagents/additives: as reported or appropriate")

    return ", ".join(parts)


def make_route_json(row: Dict[str, Any]) -> Dict[str, Any]:
    reagents = "; ".join(row["reagents"]) if row["reagents"] else "not specified"
    solvent = "; ".join(row["solvents"]) if row["solvents"] else "not specified"

    if row["yield_percent"] is None:
        yield_text = "not specified"
    else:
        yield_text = f"{round(float(row['yield_percent']), 1)}%"

    evidence_ids = [row["reaction_id"]] if row["reaction_id"] else []

    return {
        "routes": [
            {
                "route_name": "Single-step ORD-derived route",
                "summary": "One single-step forward synthesis proposal derived from a structured ORD reaction record.",
                "steps": [
                    {
                        "reaction_name": "ORD evidence-based single-step synthesis",
                        "reactants": row["reactants"],
                        "product_smiles": row["product_smiles"],
                        "stoichiometry": build_stoichiometry(row),
                        "reagents": reagents,
                        "solvent": solvent,
                        "temperature_celsius": row["temperature_celsius"],
                        "reaction_time": row["reaction_time"],
                        "atmosphere": row["atmosphere"],
                        "workup_purification": row["workup_purification"],
                        "expected_yield_percent": yield_text,
                        "important_conditions": row["important_conditions"],
                        "rationale": "The proposed single reaction is based on an ORD record where the listed reactants and conditions are associated with formation of the target product.",
                        "objective_fit": "This route is selected as a practical single-step recommendation supported by the available ORD evidence.",
                        "evidence_reaction_ids": evidence_ids,
                    }
                ],
                "objective_fit": "This route is selected because it is a direct single-step ORD-derived proposal for the target product.",
                "evidence_reaction_ids": evidence_ids,
            }
        ],
        "overall_recommendation": "Single-step ORD-derived route is recommended because it directly matches the required one-step format and uses evidence-backed reaction conditions.",
    }


def make_user_prompt(row: Dict[str, Any]) -> str:
    if USE_REACTANTS_IN_PROMPT:
        return f"""Target Molecule SMILES: {row["product_smiles"]}
Known reactants: {json.dumps(row["reactants"], ensure_ascii=False)}
Every route must contain exactly one reaction step.
Every route's only step product_smiles must be this exact target molecule.
Generate exactly 1 distinct route option.
Return valid JSON only."""

    return f"""Target Molecule SMILES: {row["product_smiles"]}
Known reactants are not provided.
Propose one plausible single-step route to make the target molecule.
Every route must contain exactly one reaction step.
Every route's only step product_smiles must be this exact target molecule.
Generate exactly 1 distinct route option.
Return valid JSON only."""


def make_sft_record(row: Dict[str, Any]) -> Dict[str, Any]:
    assistant_json = make_route_json(row)

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT_TRAIN,
            },
            {
                "role": "user",
                "content": make_user_prompt(row),
            },
            {
                "role": "assistant",
                "content": json.dumps(
                    assistant_json,
                    ensure_ascii=False,
                    separators=(",", ":"),
                ),
            },
        ]
    }


def write_jsonl(path: Path, records: Iterable[Dict[str, Any]]) -> int:
    count = 0

    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1

    return count


# Main: download + parse + write JSONL

if TRAIN_JSONL.exists() and EVAL_JSONL.exists() and not FORCE_REBUILD_DATASET:
    print("Prepared dataset already exists. Skipping rebuild.")
    print("Train:", TRAIN_JSONL)
    print("Eval:", EVAL_JSONL)

else:
    api = HfApi(token=HF_TOKEN)

    print("Listing ORD dataset files...")

    repo_files = api.list_repo_files(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
    )

    all_pb_paths = sorted(
        [
            f for f in repo_files
            if f.startswith("data/") and f.endswith(".pb.gz")
        ]
    )

    if not all_pb_paths:
        raise RuntimeError("No ORD .pb.gz files found in Hugging Face dataset repo.")

    random.seed(SEED)
    random.shuffle(all_pb_paths)

    selected_pb_paths = (
        all_pb_paths[:MAX_PB_FILES]
        if MAX_PB_FILES is not None
        else all_pb_paths
    )

    print("Total ORD pb.gz files:", len(all_pb_paths))
    print("Selected pb.gz files:", len(selected_pb_paths))

    local_pb_files = []

    for filename in tqdm(selected_pb_paths, desc="Downloading ORD pb.gz files"):
        local_path = hf_hub_download(
            repo_id=HF_DATASET_ID,
            repo_type="dataset",
            filename=filename,
            local_dir=str(ORD_DIR),
            token=HF_TOKEN,
        )

        local_pb_files.append(Path(local_path))

    rows = []
    parse_errors = 0

    for pb_file in tqdm(local_pb_files, desc="Parsing ORD pb.gz files"):
        try:
            for reaction in iter_ord_reactions_from_pb_gz(pb_file):
                row = extract_reaction_row(reaction)

                if row is not None:
                    rows.append(row)

        except Exception as exc:
            parse_errors += 1
            print("Parse failed:", pb_file.name, exc)

    print("Raw extracted rows:", len(rows))
    print("Parse errors:", parse_errors)

    if not rows:
        raise RuntimeError("No usable rows extracted. Increase MAX_PB_FILES.")

    if DEDUP_BY_PRODUCT:
        before = len(rows)
        rows = deduplicate_rows_by_product(rows)
        print(f"Deduplicated by product: {before} -> {len(rows)}")

    random.shuffle(rows)

    if MAX_TOTAL_EXAMPLES is not None:
        rows = rows[:MAX_TOTAL_EXAMPLES]

    with ROWS_JSONL.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    records = [make_sft_record(row) for row in rows]

    eval_size = min(MAX_EVAL_EXAMPLES, max(1, int(0.05 * len(records))))

    eval_records = records[:eval_size]
    train_records = records[eval_size:]

    train_count = write_jsonl(TRAIN_JSONL, train_records)
    eval_count = write_jsonl(EVAL_JSONL, eval_records)

    print("Train records:", train_count)
    print("Eval records:", eval_count)
    print("Rows saved:", ROWS_JSONL)
    print("Train JSONL:", TRAIN_JSONL)
    print("Eval JSONL:", EVAL_JSONL)

    print("\nExample SFT record:")
    print(json.dumps(train_records[0], ensure_ascii=False, indent=2)[:3000])

Listing ORD dataset files...
Total ORD pb.gz files: 550
Selected pb.gz files: 120


data/23/ord_dataset-2369b9b9f44641b19305(…):   0%|          | 0.00/923k [00:00<?, ?B/s]

data/46/ord_dataset-46d216f2c4374ef5b9df(…):   0%|          | 0.00/4.63M [00:00<?, ?B/s]

data/4a/ord_dataset-4ad5db8537994579bef5(…):   0%|          | 0.00/2.86M [00:00<?, ?B/s]

data/ac/ord_dataset-acbdbaa766314b66a982(…):   0%|          | 0.00/906k [00:00<?, ?B/s]

...

data/56/ord_dataset-56a22bc0c3b14f87b9aa(…):   0%|          | 0.00/1.84M [00:00<?, ?B/s]

data/9b/ord_dataset-9b8aa9a7835143ef8ce3(…):   0%|          | 0.00/91.7k [00:00<?, ?B/s]

data/31/ord_dataset-31641fb65b34430fa743(…):   0%|          | 0.00/5.34M [00:00<?, ?B/s]

Parsing ORD pb.gz files:   0%|          | 0/120 [00:00<?, ?it/s]

Raw extracted rows: 396580
Parse errors: 0
Deduplicated by product: 396580 -> 265353
Train records: 29200
Eval records: 800
Rows saved: /content/ord_retro_training/prepared_data/extracted_rows.jsonl
Train JSONL: /content/ord_retro_training/prepared_data/train_sft.jsonl
Eval JSONL: /content/ord_retro_training/prepared_data/eval_sft.jsonl

Example SFT record:
{
  "messages": [
    {
      "role": "system",
      "content": "You are an expert Organic Chemist specializing in Retrosynthesis.\nYour task is to propose practical single-step forward synthesis routes that make the given target molecule.\n\nRules:\n1. Output valid JSON only.\n2. Generate exactly 1 route.\n3. Each route must contain exactly 1 reaction step.\n4. The step product_smiles must equal the target molecule SMILES.\n5. Use standard SMILES strings.\n6. Use realistic organic chemistry conditions.\n7. Do not output Markdown.\n8. If evidence reaction IDs are known, include them. Do not invent unsupported evidence."
    },
    

In [6]:
# CELL 4 — Load tokenizer and render chat dataset

from datasets import load_dataset
from transformers import AutoTokenizer

assert TRAIN_JSONL.exists(), f"Missing {TRAIN_JSONL}"
assert EVAL_JSONL.exists(), f"Missing {EVAL_JSONL}"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

raw_dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_JSONL),
        "eval": str(EVAL_JSONL),
    },
)

def render_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        text += tokenizer.eos_token

    return {"text": text}


rendered_train = raw_dataset["train"].map(
    render_text,
    remove_columns=raw_dataset["train"].column_names,
)

rendered_eval = raw_dataset["eval"].map(
    render_text,
    remove_columns=raw_dataset["eval"].column_names,
)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )


train_dataset = rendered_train.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    num_proc=4,
)

eval_dataset = rendered_eval.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    num_proc=4,
)

print("Train rows:", len(train_dataset))
print("Eval rows:", len(eval_dataset))
print("Example token length:", len(train_dataset[0]["input_ids"]))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Map:   0%|          | 0/29200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/29200 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/800 [00:00<?, ? examples/s]

Train rows: 29200
Eval rows: 800
Example token length: 790


In [7]:
# CELL 5 — Load Qwen in 4-bit and apply LoRA

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.torch_dtype = torch.bfloat16

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

dtype_counts = {}

for name, param in model.named_parameters():
    if param.requires_grad:
        dtype_counts[str(param.dtype)] = dtype_counts.get(str(param.dtype), 0) + param.numel()

print("Trainable dtype counts:", dtype_counts)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364
Trainable dtype counts: {'torch.float32': 36929536}


In [8]:
# CELL 6 — Train

from transformers import Trainer, TrainingArguments


def causal_lm_data_collator(features):
    batch = tokenizer.pad(
        features,
        padding=True,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels

    return batch


WARMUP_STEPS = max(1, int(MAX_STEPS * 0.05))

common_args = dict(
    output_dir=str(OUTPUT_DIR),

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    weight_decay=0.01,

    bf16=USE_BF16,
    fp16=USE_FP16,

    logging_steps=10,

    eval_steps=250,
    save_steps=250,
    save_total_limit=3,

    report_to="none",
    seed=SEED,

    remove_unused_columns=False,
    gradient_checkpointing=True,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

try:
    training_args = TrainingArguments(
        **common_args,
        eval_strategy="steps",
        save_strategy="steps",
    )
except TypeError:
    training_args = TrainingArguments(
        **common_args,
        evaluation_strategy="steps",
        save_strategy="steps",
    )


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=causal_lm_data_collator,
    processing_class=tokenizer,
)

checkpoint_dirs = sorted(OUTPUT_DIR.glob("checkpoint-*"))

if checkpoint_dirs:
    print("Found checkpoints:")
    for ckpt in checkpoint_dirs:
        print(" -", ckpt)

    latest_checkpoint = str(checkpoint_dirs[-1])
    print("Resuming from:", latest_checkpoint)

    train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    train_result = trainer.train()

print(train_result)

eval_result = trainer.evaluate()
print(eval_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
250,0.378492,0.373053
500,0.354378,0.352592
750,0.344822,0.344703
1000,0.340078,0.337277
1250,0.344931,0.332213
1500,0.338050,0.327184
1750,0.333734,0.323003
2000,0.310835,0.320977
2250,0.317970,0.319029
2500,0.310608,0.318637


TrainOutput(global_step=2500, training_loss=0.3605717908859253, metrics={'train_runtime': 8985.0778, 'train_samples_per_second': 4.452, 'train_steps_per_second': 0.278, 'total_flos': 2.3967754788652646e+17, 'train_loss': 0.3605717908859253, 'epoch': 1.36986301369863})


Training Loss,Validation Loss,Step
0.310608,0.318637,2500


{'eval_loss': 0.3186369836330414}


In [9]:
# CELL 7 — Save and upload LoRA adapter

from huggingface_hub import HfApi

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

adapter_readme = f"""---
license: apache-2.0
base_model: {BASE_MODEL}
tags:
- chemistry
- retrosynthesis
- organic-chemistry
- qlora
- lora
- ord
---

# {HF_MODEL_REPO_NAME} LoRA adapter

QLoRA adapter fine-tuned on ORD-derived single-step reaction records.

Task:

target molecule SMILES -> one single-step JSON synthesis proposal.

Training setup:

- Base model: {BASE_MODEL}
- Max sequence length: {MAX_SEQ_LENGTH}
- Max steps: {MAX_STEPS}
- LoRA rank: 32
- BF16: {USE_BF16}

Limitations:

- Academic/diploma demonstration only.
- Not authoritative chemistry guidance.
- Product-only retrosynthesis is under-constrained.
- Generated routes must be manually checked.
"""

(ADAPTER_DIR / "README.md").write_text(adapter_readme, encoding="utf-8")

api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=LORA_REPO_ID,
    repo_type="model",
    exist_ok=True,
)

api.upload_folder(
    folder_path=str(ADAPTER_DIR),
    repo_id=LORA_REPO_ID,
    repo_type="model",
)

print("Uploaded LoRA adapter:", LORA_REPO_ID)

Uploaded LoRA adapter: oleh13/ord-retro-qwen25-15b-lora


In [11]:
# CELL 8 — Test generation

from rdkit import Chem


SYSTEM_PROMPT_TRAIN = """
You are an expert Organic Chemist specializing in Retrosynthesis.
Your task is to propose practical single-step forward synthesis routes that make the given target molecule.

Rules:
1. Output valid JSON only.
2. Generate exactly 1 route.
3. Each route must contain exactly 1 reaction step.
4. The step product_smiles must equal the target molecule SMILES.
5. Use standard SMILES strings.
6. Use realistic organic chemistry conditions.
7. Do not output Markdown.
8. If evidence reaction IDs are known, include them. Do not invent unsupported evidence.
""".strip()


def canonical_smiles(smiles: Optional[str]) -> Optional[str]:
    if not smiles:
        return None

    mol = Chem.MolFromSmiles(str(smiles).strip())

    if mol is None:
        return None

    return Chem.MolToSmiles(mol, canonical=True)


def extract_json_object(text: str) -> str:
    text = text.strip()
    first = text.find("{")
    last = text.rfind("}")

    if first == -1 or last == -1 or last <= first:
        raise ValueError("No JSON object found in model output")

    return text[first:last + 1]


def generate_route(target_smiles: str, max_new_tokens: int = 1400) -> Dict[str, Any]:
    target = canonical_smiles(target_smiles)

    if target is None:
        raise ValueError(f"Invalid target SMILES: {target_smiles}")

    user_prompt = f"""Target Molecule SMILES: {target}
Known reactants are not provided.
Propose one plausible single-step route to make the target molecule.
Every route must contain exactly one reaction step.
Every route's only step product_smiles must be this exact target molecule.
Generate exactly 1 distinct route option.
Return valid JSON only."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TRAIN},
        {"role": "user", "content": user_prompt},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    clean = extract_json_object(generated)

    return json.loads(clean)


test_target = "N#CCc1ccccc1"

result = generate_route(test_target)

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "routes": [
    {
      "route_name": "Single-step ORD-derived route",
      "summary": "One single-step forward synthesis proposal derived from a structured ORD reaction record.",
      "steps": [
        {
          "reaction_name": "ORD evidence-based single-step synthesis",
          "reactants": [
            "N#CCc1ccccc1.Cl",
            "[H][H]"
          ],
          "product_smiles": "N#CCc1ccccc1",
          "stoichiometry": "Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv, reagents/additives: as reported or appropriate",
          "reagents": "palladium on carbon",
          "solvent": "ethanol",
          "temperature_celsius": "not specified",
          "reaction_time": "not specified",
          "atmosphere": "not specified",
          "workup_purification": "FILTRATION: The catalyst was filtered off; CONCENTRATION: the filtrate was concentrated under reduced pressure; CUSTOM: The residue was purified by silica gel column chromatography (hexane/ethyl acetate=100/0→90/10

In [12]:
# CELL 9 — Merge LoRA into FP16 HF model

!python -m pip uninstall -y -q torchao

import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

try:
    del trainer
except Exception:
    pass

try:
    del model
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()

MERGED_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

merged_model = PeftModel.from_pretrained(
    base_model_fp16,
    str(ADAPTER_DIR),
    torch_dtype=torch.float16,
)

merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(
    str(MERGED_DIR),
    safe_serialization=True,
    max_shard_size="2GB",
)

tokenizer.save_pretrained(str(MERGED_DIR))

print("Merged model saved to:", MERGED_DIR)

for p in MERGED_DIR.iterdir():
    print(" -", p.name)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merged model saved to: /content/ord_retro_training/merged_fp16
 - model-00002-of-00002.safetensors
 - tokenizer_config.json
 - model.safetensors.index.json
 - tokenizer.json
 - config.json
 - generation_config.json
 - chat_template.jinja
 - model-00001-of-00002.safetensors


In [13]:
# CELL 10 — Upload merged FP16 model to HF

from huggingface_hub import HfApi

assert MERGED_DIR.exists(), f"Missing merged model folder: {MERGED_DIR}"

api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=MERGED_REPO_ID,
    repo_type="model",
    exist_ok=True,
)

api.upload_folder(
    folder_path=str(MERGED_DIR),
    repo_id=MERGED_REPO_ID,
    repo_type="model",
)

print("Uploaded merged FP16 model:", MERGED_REPO_ID)

Uploaded merged FP16 model: oleh13/ord-retro-qwen25-15b-merged-fp16
